In [5]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "bert-base-multilingual-cased"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_model_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=20,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

print("Final Text Model Accuracy:", results.metrics["test_accuracy"])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_17246/2330476936.py:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting Text Model Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 